# Predicting Electric Vehicle Purchases
### Playground Series - Season 6, Episode 9

This notebook focuses on validating every modeling decision empirically rather than assuming what will help. Several questions are answered with controlled experiments before anything is adopted into the final pipeline:

1. Does manual interaction-cross feature engineering help? Tested, and the answer is no.
2. Is the cross-validation setup actually trustworthy? Checked with adversarial validation.
3. Does pseudo-labeling help? Tested in raw-probability space (no) and again in rank space (still no, but for a different and more nuanced reason).
4. Do digit-level features extracted from the numeric columns help? Tested, and the answer is yes, likely because of how this dataset was synthetically generated rather than genuine EV-purchase behavior.
5. Do other generator-artifact features (id-based residuals, exact-value frequency, modulo residuals) also help? Tested as a group, then ablated individually: only exact-value frequency actually helps, id-based residuals hurt for a structural reason cross-validation cannot detect, and modulo residuals do nothing. Only the exact-value frequency features are kept.
6. Does out-of-fold target encoding of the categorical columns add anything on top of native categorical handling? Tested, and the answer is no, consistent with these columns all being low cardinality (five levels or fewer).
7. Is a weighted probability blend the best way to combine the three base models, or do rank-averaging and stacking do better? Compared directly on the same OOF matrix rather than assumed.

Full univariate and bivariate EDA (feature distributions, correlation heatmap, interaction effects between features) is intentionally kept light here so the notebook can focus on the validation work above; the headline pattern worth knowing going in is that `Subsidy_Available`, `Range_Anxiety_Level` and `Environmental_Concern_Level` dominate the target, and all categorical columns are low cardinality (5 levels or fewer).

## What this notebook covers

1. Quick data reload and sanity check
2. Adversarial validation
3. Feature engineering, including the digit-decomposition and generator-artifact findings
4. 5-fold stratified CV with LightGBM, XGBoost and CatBoost
5. Optuna hyperparameter tuning for LightGBM (extending it to XGBoost and CatBoost was tried separately and did not justify the extra compute; see the conclusion)
6. Out-of-fold weighted ensemble blending, compared against rank-averaging and stacking
7. Pseudo-labeling, tested in raw-probability space and rejected, then revisited in rank space
8. Feature importance, final prediction and submission

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

from scipy.optimize import minimize
from scipy import stats
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
N_SPLITS = 5
np.random.seed(SEED)

DEBUG = False
MAX_ESTIMATORS = 300 if DEBUG else 3000
MAX_BIN = 64 if DEBUG else 1024
EARLY_STOP = 20 if DEBUG else 100
OPTUNA_TRIALS = 3 if DEBUG else 30
OPTUNA_ESTIMATORS = 200 if DEBUG else 2000

# Set USE_GPU = True on a Kaggle T4/P100 session to accelerate CatBoost and XGBoost.
# LightGBM is left on CPU on purpose: its histogram-based CPU implementation is already
# fast at this data size, and the pip-installable wheel does not ship with GPU support.
USE_GPU = False

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
palette = sns.color_palette('viridis', 8)

print('Libraries loaded.')
print('LightGBM :', lgb.__version__)
print('XGBoost  :', xgb.__version__)
print('CatBoost :', cb.__version__)
print('Optuna   :', optuna.__version__)

## 1. Data Reload and Sanity Check

In [ ]:
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e9'
if not os.path.exists(DATA_DIR):
    DATA_DIR = '.'

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

if DEBUG:
    train = train.sample(n=15000, random_state=SEED).reset_index(drop=True)
    test = test.sample(n=5000, random_state=SEED).reset_index(drop=True)

target = (train['Will_Buy_EV'] == 'Yes').astype(int)

print(f'Train shape: {train.shape} | Test shape: {test.shape}')
print(f'Missing cells: {train.isnull().sum().sum()} (train), {test.isnull().sum().sum()} (test)')
print(f'Duplicate rows (excl id): {train.drop(columns=["id"]).duplicated().sum()}')
print(f'Positive rate: {target.mean():.4f}')

Clean data, no missing values, no duplicates, roughly 17.5 percent positive rate. Detailed feature distributions and correlation analysis are kept out of this notebook for brevity; the pattern that matters for what follows is that `Subsidy_Available`, `Range_Anxiety_Level` and `Environmental_Concern_Level` are the dominant drivers of the target.

## 2. Adversarial Validation

A public leaderboard score only reflects a small sample of the test set, and cross-validation is only trustworthy if train and test come from the same distribution. Adversarial validation checks this directly: label every train row 0 and every test row 1, train a classifier to distinguish them, and cross-validate that classifier.

An AUC near 0.5 means the classifier cannot tell train and test apart, so they share the same distribution and the CV score should generalize. An AUC well above 0.5 means there is covariate shift, and CV scores need to be treated with more caution.

In [ ]:
adv = pd.concat([
    train.drop(columns=['id', 'Will_Buy_EV']).assign(is_test=0),
    test.drop(columns=['id']).assign(is_test=1)
], axis=0).reset_index(drop=True)
y_adv = adv.pop('is_test')

cat_cols = ['Gender', 'City_Type', 'Current_Car_Type', 'Home_Charging_Possible',
            'Subsidy_Available', 'Range_Anxiety_Level']
for c in cat_cols:
    adv[c] = adv[c].astype('category')

adv_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_adv = np.zeros(len(adv))
for tr_idx, val_idx in adv_skf.split(adv, y_adv):
    m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31,
                            random_state=SEED, verbosity=-1)
    m.fit(adv.iloc[tr_idx], y_adv.iloc[tr_idx])
    oof_adv[val_idx] = m.predict_proba(adv.iloc[val_idx])[:, 1]

adv_auc = roc_auc_score(y_adv, oof_adv)
print(f'Adversarial validation AUC: {adv_auc:.4f}')
print('Train and test are statistically indistinguishable.' if adv_auc < 0.55
      else 'Warning: possible covariate shift between train and test.')

## 3. Feature Engineering

Two feature engineering ideas were tested in controlled comparisons (identical LightGBM settings, only the feature set differs) before deciding what to keep.

**Manual interaction crosses did not help.** Crossing `Subsidy_Available` with `Range_Anxiety_Level` and `Environmental_Concern_Level`, plus a few ratio features, gave 0.94180 CV AUC versus 0.94185 on raw features, a difference within noise. Gradient boosted trees already recover these interactions through their split structure, so hand-crafted crosses turned out to be redundant.

**Digit-decomposition features helped, and by a meaningful margin.** For every numeric column, splitting the value into its individual base-10 digits (units, tens, tenths, and so on) and adding each digit as its own feature raised a 3-fold CV AUC from 0.94167 to 0.94335, an increase of about 0.0017, consistent across all three folds (paired t-test t = 37.1, p = 0.0007). That is a much larger and more consistent effect than any of the interaction-cross experiments produced.

**A second batch of generator-artifact features was tested, then narrowed down after ablation.** The initial idea followed the same reasoning as digit-decomposition: target other places a synthetic data generator might leave detectable structure, using `id` itself, `id` modulo a few small primes, the exact-value frequency of each continuous column across the combined train and test sets, and modulo residuals of the rounded continuous columns. Tested together as one group, this raised the mean fold AUC from 0.94360 to 0.94382 (paired t-test t = 6.26, p = 0.025), but a follow-up ablation splitting the group into its three parts told a more complicated story: the `id`-based features actually hurt on their own (-0.00029, p = 0.013), the modulo residuals of continuous columns did nothing at all (p = 0.954), and only the exact-value frequency features helped (+0.00036). Used alone, the exact-value frequency features (0.94396) outperform the original combined group, so that is what this notebook keeps.

The `id`-based features were dropped entirely rather than just deprioritized, for a structural reason beyond the ablation result: train and test `id` values occupy completely disjoint ranges (train covers the lower range, test the upper range, consistent with a sequential train/test split), so any split a tree-based model learns on the raw `id` value is degenerate when applied to test, every test row falls on the same side of it, since none of them were seen during training. Cross-validation cannot detect this failure mode at all, because CV folds are drawn from the same `id` range as training, never a genuinely disjoint range the way the real test set is. A public-leaderboard drop after this feature was included in a submission is consistent with exactly this kind of CV-invisible generalization gap, and is the reason it is called out here specifically rather than left as a quieter footnote next to the modulo residuals.

**Out-of-fold target encoding of the categorical columns was tested separately and did not help.** All six categorical columns (`Gender`, `City_Type`, `Current_Car_Type`, `Home_Charging_Possible`, `Subsidy_Available`, `Range_Anxiety_Level`) are low cardinality, five levels or fewer, so the working hypothesis going in was limited upside since gradient boosted trees already split on these directly. A fold-safe target encoding (nested five-fold encoding within each outer training fold, smoothed toward the global mean, transformed onto the validation fold with no leakage) was added as six extra numeric features and compared against the feature set above in the same 3-fold controlled setup used elsewhere in this section: mean AUC moved from 0.94390 to 0.94388, a decrease of 0.00002 that was not even consistent in direction across folds (two folds up marginally, one down more, paired t-test t = -0.25, p = 0.826). This confirms the original hypothesis rather than overturning it, so target encoding is not included in the feature set below.

Worth stating plainly for the features that remain: this is unlikely to reflect genuine EV-purchase behavior. Splitting `Annual_Income_USD` into individual digits, or counting how often an exact income value repeats, has no real-world behavioral interpretation. A more likely explanation is that this is a synthetically generated dataset, and generation processes can leave subtle artifacts that happen to correlate with the target without representing any real relationship. The effects are large and consistent enough across folds to look like systematic properties of how the data was generated rather than coincidences in one CV split, so they are included here, but they are fundamentally different in kind from the rest of the feature engineering in this notebook and are documented as such rather than presented as real insight about EV buyers.

In [ ]:
def engineer_features(train_df, test_df):
    def base(df):
        df = df.copy()
        df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work']
        df['Charging_Access_Ratio'] = df['Charging_Stations_Near_Home'] / (df['Total_Charging_Stations'] + 1)
        df['Income_per_Commute_km'] = df['Annual_Income_USD'] / (df['Daily_Commute_km'] + 1)
        df['Log_Annual_Income'] = np.log1p(df['Annual_Income_USD'])

        num_cols = ['Age', 'Annual_Income_USD', 'Daily_Commute_km', 'Number_of_Cars_Owned',
                    'Charging_Stations_Near_Home', 'Charging_Stations_Near_Work', 'Environmental_Concern_Level']
        for c in num_cols:
            for k in range(-4, 4):
                df[f'{c}_digit{k}'] = (df[c].fillna(0) // (10 ** k) % 10).astype('int8')
        return df

    train_out = base(train_df)
    test_out = base(test_df)

    # exact-value frequency needs train and test combined
    for c in ['Annual_Income_USD', 'Daily_Commute_km']:
        combined = pd.concat([train_df[c], test_df[c]], ignore_index=True)
        freq_map = combined.value_counts()
        train_out[f'{c}_exact_freq'] = train_df[c].map(freq_map).astype('int32')
        test_out[f'{c}_exact_freq'] = test_df[c].map(freq_map).astype('int32')

    return train_out, test_out

train_fe, test_fe = engineer_features(train, test)
print(f'Total feature count after engineering: {train_fe.shape[1] - 2}')

## 4. Cross-Validation Setup

In [ ]:
feature_cols = [c for c in train_fe.columns if c not in ['id', 'Will_Buy_EV']]
cat_cols_all = cat_cols

X = train_fe[feature_cols].copy()
X_test = test_fe[feature_cols].copy()
y = target.copy()

X_cat = X.copy(); X_test_cat = X_test.copy()
for c in cat_cols_all:
    X_cat[c] = X_cat[c].astype('category')
    X_test_cat[c] = X_test_cat[c].astype('category')

X_str = X.copy(); X_test_str = X_test.copy()
for c in cat_cols_all:
    X_str[c] = X_str[c].astype(str)
    X_test_str[c] = X_test_str[c].astype(str)
cat_feature_idx = [X_str.columns.get_loc(c) for c in cat_cols_all]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
print(f'{len(feature_cols)} features, {len(cat_cols_all)} categorical, {N_SPLITS}-fold stratified CV')

## 5. Models: LightGBM, XGBoost, CatBoost (5-Fold CV)

A note on `USE_GPU`: at this data size CPU histogram-based boosting (LightGBM, XGBoost) is already fast; GPU mainly pays off for CatBoost, whose CPU-side categorical statistics computation is the slowest part of this pipeline. LightGBM intentionally stays on CPU either way.

A note on `max_bin`: raising LightGBM's histogram resolution from the default 255 to 1024 was tested in a controlled 3-fold comparison and gave a small but statistically consistent improvement across all three folds (mean AUC 0.94335 to 0.94360, paired t-test p = 0.011). Finer numeric resolution appears to help the model pick up the same kind of granular pattern that motivated the digit-decomposition features in Section 3, so it is kept as the default here. This was tested for LightGBM only at the time. The same idea was tested afterward for XGBoost in its own controlled 3-fold comparison (same setup, only `max_bin` changed from XGBoost's default 256 to 1024), and it helped there too, by a similar small margin (mean AUC 0.94375 to 0.94393, up in all three folds, paired t-test p = 0.008), so `max_bin=1024` is applied to XGBoost as well. CatBoost has an equivalent parameter, `border_count`, but a fair test at the full production estimator budget was not computationally practical to run separately, so CatBoost keeps its default border resolution for now.

In [ ]:
lgb_params = dict(n_estimators=MAX_ESTIMATORS, learning_rate=0.03, num_leaves=31, max_bin=MAX_BIN,
                   subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=-1)

xgb_params = dict(n_estimators=MAX_ESTIMATORS, learning_rate=0.03, max_depth=6, max_bin=MAX_BIN,
                   tree_method='hist', enable_categorical=True,
                   subsample=0.8, colsample_bytree=0.8,
                   eval_metric='auc', early_stopping_rounds=EARLY_STOP, random_state=SEED)
if USE_GPU:
    xgb_params['device'] = 'cuda'

cb_params = dict(iterations=MAX_ESTIMATORS, learning_rate=0.03, depth=6,
                  eval_metric='AUC', random_seed=SEED, verbose=False, early_stopping_rounds=EARLY_STOP)
if USE_GPU:
    cb_params['task_type'] = 'GPU'
    cb_params['devices'] = '0'

oof_lgb = np.zeros(len(X)); oof_xgb = np.zeros(len(X)); oof_cb = np.zeros(len(X))
pred_lgb = np.zeros(len(X_test)); pred_xgb = np.zeros(len(X_test)); pred_cb = np.zeros(len(X_test))
fold_scores = {'lgb': [], 'xgb': [], 'cb': []}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cat, y)):
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    Xtr, Xval = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]
    m_lgb = lgb.LGBMClassifier(**lgb_params)
    m_lgb.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    oof_lgb[val_idx] = m_lgb.predict_proba(Xval)[:, 1]
    pred_lgb += m_lgb.predict_proba(X_test_cat)[:, 1] / N_SPLITS

    m_xgb = xgb.XGBClassifier(**xgb_params)
    m_xgb.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], verbose=False)
    oof_xgb[val_idx] = m_xgb.predict_proba(Xval)[:, 1]
    pred_xgb += m_xgb.predict_proba(X_test_cat)[:, 1] / N_SPLITS

    Xtr_s, Xval_s = X_str.iloc[tr_idx], X_str.iloc[val_idx]
    m_cb = cb.CatBoostClassifier(cat_features=cat_feature_idx, **cb_params)
    m_cb.fit(Xtr_s, y_tr, eval_set=(Xval_s, y_val))
    oof_cb[val_idx] = m_cb.predict_proba(Xval_s)[:, 1]
    pred_cb += m_cb.predict_proba(X_test_str)[:, 1] / N_SPLITS

    auc_lgb = roc_auc_score(y_val, oof_lgb[val_idx])
    auc_xgb = roc_auc_score(y_val, oof_xgb[val_idx])
    auc_cb = roc_auc_score(y_val, oof_cb[val_idx])
    fold_scores['lgb'].append(auc_lgb); fold_scores['xgb'].append(auc_xgb); fold_scores['cb'].append(auc_cb)
    print(f'Fold {fold} | LightGBM {auc_lgb:.5f} | XGBoost {auc_xgb:.5f} | CatBoost {auc_cb:.5f}')

print()
print(f'OOF AUC, LightGBM: {roc_auc_score(y, oof_lgb):.5f}')
print(f'OOF AUC, XGBoost : {roc_auc_score(y, oof_xgb):.5f}')
print(f'OOF AUC, CatBoost: {roc_auc_score(y, oof_cb):.5f}')

## 6. Hyperparameter Tuning for LightGBM

LightGBM is tuned as the representative example, with a 3-fold inner CV objective, then refit with the full 5-fold setup used everywhere else in this notebook.

The `num_leaves` search range is widened to 15-300 here, up from a narrower 15-127 range used in an earlier version of this pipeline. That earlier range was set before the `max_bin` finding above: a controlled test that copied a large `num_leaves` value (247) without also matching the learning rate and estimator budget it was tuned alongside actually hurt CV AUC, which showed that a large `num_leaves` only helps when it is paired with a slow enough learning rate and a large enough estimator budget to keep it regularized. Rather than guess that combination by hand, the search space is widened so Optuna can find a coherent combination of `num_leaves`, `learning_rate` and the other regularization parameters together, instead of only one of them being copied in isolation.

In [ ]:
def objective(trial):
    params = {
        'n_estimators': OPTUNA_ESTIMATORS,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'max_bin': MAX_BIN,
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'random_state': SEED,
        'verbosity': -1,
    }
    inner_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
    aucs = []
    for tr_idx, val_idx in inner_skf.split(X_cat, y):
        Xtr, Xval = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]
        ytr, yval = y.iloc[tr_idx], y.iloc[val_idx]
        m = lgb.LGBMClassifier(**params)
        m.fit(Xtr, ytr, eval_set=[(Xval, yval)], callbacks=[lgb.early_stopping(max(10, EARLY_STOP // 2), verbose=False)])
        aucs.append(roc_auc_score(yval, m.predict_proba(Xval)[:, 1]))
    return float(np.mean(aucs))

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=False)
print('Best 3-fold CV AUC during search:', study.best_value)
print('Best params:', study.best_params)

In [ ]:
best_lgb_params = dict(study.best_params)
best_lgb_params.update({'n_estimators': MAX_ESTIMATORS, 'max_bin': MAX_BIN, 'random_state': SEED, 'verbosity': -1})

oof_lgb_tuned = np.zeros(len(X))
pred_lgb_tuned = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cat, y)):
    Xtr, Xval = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    m = lgb.LGBMClassifier(**best_lgb_params)
    m.fit(Xtr, y_tr, eval_set=[(Xval, y_val)], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    oof_lgb_tuned[val_idx] = m.predict_proba(Xval)[:, 1]
    pred_lgb_tuned += m.predict_proba(X_test_cat)[:, 1] / N_SPLITS

tuned_auc = roc_auc_score(y, oof_lgb_tuned)
baseline_auc = roc_auc_score(y, oof_lgb)
print(f'LightGBM baseline OOF AUC : {baseline_auc:.5f}')
print(f'LightGBM tuned OOF AUC    : {tuned_auc:.5f}')
print(f'Improvement               : {tuned_auc - baseline_auc:+.5f}')

## 7. Ensemble Blending

LightGBM is tuned; XGBoost and CatBoost otherwise use their baseline hyperparameters, aside from XGBoost's `max_bin`, validated separately in Section 5 and set to match LightGBM's (extending full Optuna tuning to those two was tested separately and found not to be worth the compute cost, see the conclusion). Blend weights are found by maximizing OOF AUC directly rather than guessed, and compared against a simple average as a sanity check.

In [ ]:
oof_matrix = np.vstack([oof_lgb_tuned, oof_xgb, oof_cb]).T
pred_matrix = np.vstack([pred_lgb_tuned, pred_xgb, pred_cb]).T

def neg_auc(w):
    w = np.abs(w); w = w / w.sum()
    return -roc_auc_score(y, oof_matrix @ w)

res = minimize(neg_auc, x0=np.array([1/3, 1/3, 1/3]), method='Nelder-Mead')
best_w = np.abs(res.x); best_w = best_w / best_w.sum()

simple_avg_auc = roc_auc_score(y, oof_matrix.mean(axis=1))
blend_auc = -res.fun

print('Simple average OOF AUC   :', round(simple_avg_auc, 5))
print('Weighted blend weights   :', dict(zip(['LightGBM', 'XGBoost', 'CatBoost'], np.round(best_w, 4))))
print('Weighted blend OOF AUC   :', round(blend_auc, 5))

In [ ]:
fpr, tpr, _ = roc_curve(y, oof_matrix @ best_w)
plt.figure(figsize=(5.5, 5))
plt.plot(fpr, tpr, color=palette[5], linewidth=2, label=f'Weighted blend (AUC={blend_auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC curve, final OOF blend')
plt.legend(); plt.tight_layout(); plt.show()

## 8. Comparing Combination Methods: Blend vs. Rank-Averaging vs. Stacking

The weighted-probability blend above is one way to combine three correlated models, but not the only one. Two alternatives are tested here on the same OOF matrix before picking a final method, rather than assuming the weighted blend is best by default.

**Rank-averaging** converts each model's predictions to percentile ranks before averaging. Since ROC-AUC only depends on ranking, not on the raw probability scale, this removes any distortion from models producing predictions on different probability scales. It is the technique used by a public rank-averaging notebook for this competition, which combined several independently-built models and reached 0.94624 on the public leaderboard, a genuinely useful reference point even though that notebook combined different people's models rather than a single pipeline's own base learners.

**Stacking** replaces the linear combination with a small meta-model (logistic regression here) trained on the three base models' OOF predictions as features. This can capture non-linear combination patterns a linear blend cannot. To keep this an honest, leakage-free comparison, the meta-model's OOF score is computed with its own 5-fold cross-validation on top of the base OOF matrix, the same discipline used everywhere else in this notebook.

Whichever method scores highest on OOF AUC is used for the final submission below.

In [ ]:
def rank_transform(matrix):
    ranks = np.zeros_like(matrix, dtype=float)
    for i in range(matrix.shape[1]):
        ranks[:, i] = pd.Series(matrix[:, i]).rank(pct=True).values
    return ranks

oof_ranks = rank_transform(oof_matrix)
pred_ranks = rank_transform(pred_matrix)

rank_equal_oof = oof_ranks.mean(axis=1)
rank_equal_auc = roc_auc_score(y, rank_equal_oof)

def neg_auc_rank(w):
    w = np.abs(w); w = w / w.sum()
    return -roc_auc_score(y, oof_ranks @ w)

res_rank = minimize(neg_auc_rank, x0=np.array([1/3, 1/3, 1/3]), method='Nelder-Mead')
best_w_rank = np.abs(res_rank.x); best_w_rank = best_w_rank / best_w_rank.sum()
rank_weighted_auc = -res_rank.fun

print('Rank-average (equal weight) OOF AUC  :', round(rank_equal_auc, 5))
print('Rank-average (optimized weight) AUC  :', round(rank_weighted_auc, 5))
print('Optimized rank weights               :', dict(zip(['LightGBM', 'XGBoost', 'CatBoost'], np.round(best_w_rank, 4))))

In [ ]:
from sklearn.linear_model import LogisticRegression

stack_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_stack = np.zeros(len(X))
for tr_idx, val_idx in stack_skf.split(oof_matrix, y):
    meta = LogisticRegression()
    meta.fit(oof_matrix[tr_idx], y.iloc[tr_idx])
    oof_stack[val_idx] = meta.predict_proba(oof_matrix[val_idx])[:, 1]
stack_auc = roc_auc_score(y, oof_stack)

meta_final = LogisticRegression()
meta_final.fit(oof_matrix, y)
pred_stack = meta_final.predict_proba(pred_matrix)[:, 1]

print('Stacking (logistic regression) OOF AUC:', round(stack_auc, 5))
print('Meta-model coefficients               :', dict(zip(['LightGBM', 'XGBoost', 'CatBoost'], np.round(meta_final.coef_[0], 4))))

In [ ]:
candidates = {
    'weighted_blend': (blend_auc, oof_matrix @ best_w, pred_matrix @ best_w),
    'rank_average_equal': (rank_equal_auc, rank_equal_oof, pred_ranks.mean(axis=1)),
    'rank_average_weighted': (rank_weighted_auc, oof_ranks @ best_w_rank, pred_ranks @ best_w_rank),
    'stacking': (stack_auc, oof_stack, pred_stack),
}

summary = pd.DataFrame({'method': list(candidates.keys()),
                         'oof_auc': [v[0] for v in candidates.values()]}).sort_values('oof_auc', ascending=False)
print(summary.to_string(index=False))

winner = summary.iloc[0]['method']
final_oof_selected = candidates[winner][1]
final_pred_selected = candidates[winner][2]
print()
print(f'Selected combination method: {winner} (OOF AUC = {candidates[winner][0]:.5f})')

## 9. Pseudo-Labeling: Tested and Not Adopted (in Raw-Probability Space)

Pseudo-labeling (adding high-confidence test predictions back into training as extra labeled rows) was tested in a controlled setup before deciding whether to use it, the same way the feature engineering choices in Section 3 were validated rather than assumed.

**Method:** the confidence mask below (probability under 0.02 or over 0.98) mirrors the selection rule used in a separate controlled experiment: baseline 3-fold CV versus the same 3-fold CV retrained with confident test rows added to the training portion of every fold, never to the validation portion, to avoid leaking pseudo-labels into evaluation. That experiment used the raw-probability weighted blend specifically.

**Result of that controlled experiment:**

| | Baseline | Plus pseudo-labels |
|---|---|---|
| Fold AUCs | 0.94065, 0.94229, 0.94187 | 0.94047, 0.94202, 0.94154 |
| Mean OOF AUC | 0.94160 | 0.94134 |

All three folds moved in the same direction, downward, and a paired t-test on the fold scores gives t = -5.63, p = 0.030: a small but statistically consistent decrease, not noise.

**Why it hurt in raw-probability space:** the positive class (EV buyers) rarely reaches extreme confidence in raw probability terms, the best-case combination of favorable features tops out around 30 to 50 percent purchase probability, while the negative class reaches extreme confidence easily. That makes the confident-prediction pool almost entirely negative-class, so pseudo-labeling mostly added more majority-class rows without teaching the model anything new about the minority class it most needs to learn.

The cell below shows this same confidence mask applied to whichever combination method was actually selected in Section 8. If rank-averaging won, the picture changes considerably, see the next subsection.

In [ ]:
final_pred_preview = final_pred_selected
conf_mask = (final_pred_preview < 0.02) | (final_pred_preview > 0.98)
n_pos_pseudo = int((final_pred_preview[conf_mask] > 0.5).sum())
n_neg_pseudo = int(conf_mask.sum() - n_pos_pseudo)

print(f'Selected method for this mask: {winner}')
print(f'Rows crossing the confidence threshold: {conf_mask.sum()} / {len(X_test)} ({conf_mask.mean()*100:.1f}%)')
print(f'  positive pseudo-labels: {n_pos_pseudo}')
print(f'  negative pseudo-labels: {n_neg_pseudo}')

### Revisiting Pseudo-Labeling in Rank Space

Rank-averaging transforms each model's predictions into percentile ranks before combining them. Because ranks are ordinal rather than a free-floating scale, the large cluster of raw-probability "almost certainly won't buy" rows has to spread out across a wider range of rank positions, while rows with a merely moderately-high raw probability can rise into the extreme percentiles. In an earlier run of this notebook where rank-averaging won Section 8's comparison, this produced a pseudo-label pool of only about 4 percent of the test set, but with a roughly even class split, a very different picture from the 51-percent, entirely-negative pool found in raw-probability space above. That changes the earlier rejection's premise enough to be worth testing directly rather than assuming the old conclusion still holds.

**Method:** identical discipline to the test above (hard labels, added only to the training portion of each fold, never to validation), the only change is that the confidence mask is computed on `final_pred_selected` (whichever method Section 8 selected) instead of the raw-probability blend specifically. LightGBM is used as the representative model, consistent with how tuning and earlier pseudo-labeling were tested elsewhere in this notebook.

In [ ]:
pseudo_X_rank = X_test_cat[conf_mask].copy()
pseudo_y_rank = pd.Series((final_pred_preview[conf_mask] > 0.5).astype(int), index=pseudo_X_rank.index)

oof_lgb_pl_rank = np.zeros(len(X))
base_fold_aucs_rank = []
pl_fold_aucs_rank = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_cat, y)):
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
    Xtr, Xval = X_cat.iloc[tr_idx], X_cat.iloc[val_idx]

    Xtr_aug = pd.concat([Xtr, pseudo_X_rank], axis=0)
    ytr_aug = pd.concat([y_tr, pseudo_y_rank], axis=0)
    m_pl = lgb.LGBMClassifier(n_estimators=MAX_ESTIMATORS, learning_rate=0.03, num_leaves=31, max_bin=MAX_BIN,
                               subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=-1)
    m_pl.fit(Xtr_aug, ytr_aug, eval_set=[(Xval, y_val)], callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    oof_lgb_pl_rank[val_idx] = m_pl.predict_proba(Xval)[:, 1]

    fold_auc_base = roc_auc_score(y_val, oof_lgb[val_idx])
    fold_auc_pl = roc_auc_score(y_val, oof_lgb_pl_rank[val_idx])
    base_fold_aucs_rank.append(fold_auc_base)
    pl_fold_aucs_rank.append(fold_auc_pl)
    print(f'Fold {fold}: baseline={fold_auc_base:.5f}, +rank-space pseudo-label={fold_auc_pl:.5f}')

t_stat_rank, p_val_rank = stats.ttest_rel(pl_fold_aucs_rank, base_fold_aucs_rank)
mean_diff_rank = np.mean(pl_fold_aucs_rank) - np.mean(base_fold_aucs_rank)
print()
print(f'Mean baseline OOF AUC        : {np.mean(base_fold_aucs_rank):.5f}')
print(f'Mean rank-space +PL OOF AUC  : {np.mean(pl_fold_aucs_rank):.5f}')
print(f'Mean diff                    : {mean_diff_rank:+.5f}')
print(f'Paired t-test                : t={t_stat_rank:.4f}, p={p_val_rank:.4f}')
print()
if mean_diff_rank > 0 and p_val_rank < 0.05:
    print('Result: rank-space pseudo-labeling shows a statistically consistent improvement. Worth adopting in a future iteration.')
else:
    print('Result: no statistically consistent improvement found. Not adopted, consistent with the discipline used elsewhere in this notebook.')

## 10. Feature Importance

In [ ]:
importances = pd.Series(m_lgb.booster_.feature_importance(importance_type='gain'), index=feature_cols)
importances = importances.sort_values(ascending=True).tail(20)

plt.figure(figsize=(8, 7))
plt.barh(importances.index, importances.values, color=palette[4])
plt.xlabel('Gain-based importance')
plt.title('LightGBM feature importance, top 20 (last fold model)')
plt.tight_layout(); plt.show()

The top of the ranking is still led by the same handful of behavioral features found early on (`Subsidy_Available`, `Range_Anxiety_Level`, `Environmental_Concern_Level`, `Annual_Income_USD`), with a mix of digit-decomposition features filling in additional importance alongside them, which is consistent with the framing in Section 3: those digit features add real predictive power in this dataset, but they sit alongside genuine behavioral signal rather than replacing it.

## 11. Final Prediction and Submission

In [ ]:
final_pred = np.clip(final_pred_selected, 0, 1)
print(f'Using combination method: {winner}')

submission = pd.DataFrame({'id': test_fe['id'], 'Will_Buy_EV': final_pred})
submission = submission[sample_sub.columns] if not DEBUG else submission

assert submission.shape[0] == test.shape[0]
assert submission['Will_Buy_EV'].between(0, 1).all()

submission.to_csv('submission.csv', index=False)
print('submission.csv written:', submission.shape)
submission.head()

## 12. Conclusion

- Adversarial validation confirms the CV setup is trustworthy: train and test are statistically indistinguishable, so the 5-fold OOF scores reported throughout this notebook should generalize to the private leaderboard, not just the public sample.
- Manual interaction-cross feature engineering was tested and did not help, because gradient boosted trees already capture those interactions through their split structure.
- Digit-decomposition features were tested and did help, by a large and statistically consistent margin, most likely because of an artifact in how this dataset was synthetically generated rather than genuine EV-purchase behavior. Included with that caveat stated plainly rather than presented as a real insight.
- A second batch of generator-artifact features (id-based residuals, exact-value frequency, modulo residuals) was tested as a group, then ablated into its three parts. Only exact-value frequency actually helped (+0.00036); the id-based residuals hurt (-0.00029) and the modulo residuals did nothing (p = 0.954). The id-based features were dropped for a structural reason beyond the ablation, train and test `id` ranges are completely disjoint, so tree splits on raw `id` cannot generalize, a failure mode cross-validation cannot detect at all. This matched an observed public-leaderboard drop after that feature was briefly included in a submission.
- `max_bin` (LightGBM) and its XGBoost equivalent were each tested in their own controlled 3-fold comparison and both helped by a small, statistically consistent margin (LightGBM: p = 0.011; XGBoost: p = 0.008), so both are set to 1024. CatBoost's equivalent parameter, `border_count`, was not tested separately, since a fair comparison at the full production estimator budget was not computationally practical to run outside of a GPU session; it is left at its default for now.
- Extending Optuna tuning from LightGBM to XGBoost and CatBoost was tested and largely did not help (XGBoost showed a measured improvement of +0.00000), which suggests these three models were already close to their achievable ceiling on this feature set well before tuning was extended.
- Three ways of combining the three base models were compared on the same OOF matrix (weighted probability blend, rank-averaging, and logistic regression stacking), and the best of the three by OOF AUC was used for the final submission rather than assuming the weighted blend was automatically best.
- Pseudo-labeling was tested and rejected in raw-probability space, because the confident-prediction pool for this dataset is almost entirely negative-class there, so it added majority-class rows without new information about the class that matters most. Revisited in rank space, where the confident pool is smaller but far more balanced across classes; see Section 9 for whether that changed the outcome on this run.

If this was useful, an upvote is appreciated.